# 01 — Train / Val / Test split (70 : 15 : 15)

Loads every JSON file in `Dev Set/`, validates and cleans it, then writes a split stratified by subject to `data/splits/{train,val,test}.jsonl`.

Re-running with the same `SEED` gives exactly the same split.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

Project root: D:\SinhalaMMLU\SinhalaMMLU-Shared-Task


In [2]:
RAW_DIR   = ROOT / "Dev Set"
SPLIT_DIR = ROOT / "data" / "splits"
SEED      = 42
RATIOS    = dict(train=0.70, val=0.15, test=0.15)

## Load and validate

In [3]:
from src.data import load_raw, validate

raw = load_raw(RAW_DIR)
df, report = validate(raw)

print(f"Loaded {len(raw)} questions from {raw['source_file'].nunique()} files")
print(f"Renamed subjects: {report['renamed_subjects']}")
print(f"Repaired answers: {report['repaired']}")
print(f"Dropped ({len(report['dropped'])}): {report['dropped']}")
print(f"Clean questions: {len(df)}")
df.head()

Loaded 1851 questions from 14 files
Renamed subjects: {'Eastern_music': 'Eastern music', 'Sinhala_language_and_literature': 'Sinhala language and literature', 'drama_and_theatre': 'drama and Theatre'}
Repaired answers: [('Geography_social_science_easy_f_117::64', 24, 3)]
Dropped (0): []
Clean questions: 1851


,id,source_file,q_no,subject,category,question,choices,answer,difficulty,grade,year,metadata
0,Arts_humanities_easy_f_97::1,Arts_humanities_easy_f_97.json,1,Arts,humanities,වෙසක් උත්සවයේදී බෞද්ධයින් අනුගමනය කරන පූජා විධ...,"[ආමිස පූජා සහ ප්‍රතිපත්ති පූජා, මල් පූජා හා පහ...",1,easy,6,2022,"{'subject_original': 'චිත්‍ර', 'difficulty': '..."
1,Arts_humanities_easy_f_97::2,Arts_humanities_easy_f_97.json,2,Arts,humanities,හින්දු භක්තිකයින්ගේ ප්‍රධාන ආගමික උත්සවයකි.,"[නත්තල් උත්සවය, රාමසාන් උත්සවය, තෛපොංගල් උත්සව...",3,easy,6,2022,"{'subject_original': 'චිත්‍ර', 'difficulty': '..."
2,Arts_humanities_easy_f_97::3,Arts_humanities_easy_f_97.json,3,Arts,humanities,ඉස්ලාම් භක්තිකයින් රාමසාන් මාසයේ මුල් සඳ දුටු ...,"[නව සඳ දැකීමෙන් පසුව ය., හිරු උදාව සමගය., නව හ...",1,easy,6,2022,"{'subject_original': 'චිත්‍ර', 'difficulty': '..."
3,Arts_humanities_easy_f_97::4,Arts_humanities_easy_f_97.json,4,Arts,humanities,සෑම වසරකම දෙසැම්බර් 24 වන දින රාත්‍රීයේ දේවස්ථ...,"[නබිනායකතුමාගේ උපතයි., ජේසුස් වහන්සේගේ උපතයි.,...",2,easy,6,2022,"{'subject_original': 'චිත්‍ර', 'difficulty': '..."
4,Arts_humanities_easy_f_97::5,Arts_humanities_easy_f_97.json,5,Arts,humanities,බෞද්ධ ස්ථූප ප්‍රධාන ආකාර කීයද?,"[2, 8, 6, 10]",3,easy,6,2022,"{'subject_original': 'චිත්‍ර', 'difficulty': '..."


In [4]:
import pandas as pd

pd.DataFrame({
    "questions": df["subject"].value_counts(),
}).join(df.groupby("subject")["category"].first())

,questions,category
subject,,
science,164,stem
Buddhism,162,humanities
Sinhala language and literature,155,language
History,151,humanities
Eastern music,143,humanities
Catholicism,132,humanities
Health and Physical Science,130,social_science
Christianity,130,humanities
drama and Theatre,128,humanities


In [5]:
df["answer"].value_counts().sort_index().rename("answer label distribution")

answer
1    473
2    501
3    507
4    370
Name: answer label distribution, dtype: int64

## Split

In [6]:
from src.data import split, save_jsonl

train_df, val_df, test_df = split(df, **RATIOS, seed=SEED, stratify_col="subject")

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    save_jsonl(part, SPLIT_DIR / f"{name}.jsonl")
    print(f"{name:5s}: {len(part):5d} ({len(part)/len(df):.1%}) -> {SPLIT_DIR / (name + '.jsonl')}")

assert not (set(train_df.id) & set(val_df.id) or set(train_df.id) & set(test_df.id) or set(val_df.id) & set(test_df.id))

train:  1295 (70.0%) -> D:\SinhalaMMLU\SinhalaMMLU-Shared-Task\data\splits\train.jsonl
val  :   278 (15.0%) -> D:\SinhalaMMLU\SinhalaMMLU-Shared-Task\data\splits\val.jsonl
test :   278 (15.0%) -> D:\SinhalaMMLU\SinhalaMMLU-Shared-Task\data\splits\test.jsonl


In [7]:
# Per-subject counts in each split (should be roughly 70/15/15 in every row)
pd.DataFrame({
    "train": train_df["subject"].value_counts(),
    "val":   val_df["subject"].value_counts(),
    "test":  test_df["subject"].value_counts(),
}).fillna(0).astype(int).assign(total=lambda d: d.sum(axis=1))

,train,val,test,total
subject,,,,
Arts,70,15,15,100
Buddhism,113,24,25,162
Catholicism,92,20,20,132
Christianity,91,20,19,130
Civics,86,19,18,123
Eastern music,100,22,21,143
Geography,82,17,18,117
Health and Physical Science,91,19,20,130
History,106,23,22,151
